In [ ]:
# Imports and Constant

import numpy as np
r_earth=6371*1e3 #meters
panel_area=2 #m^2
satelite_core_area=0.5 #m^2
satelite_mass=100 # in kg
mu=3.9861e14 # gravational parameter of earth in si
rho_0=1 # in si
h_0=1e5 # 100km
scale_height=1
CD=2.2 # this is from a nasa estimate

cut_off_limit=120e3 # orbital height where we assume the satelite will burn up

In [ ]:
# Parameter helpers, definitions, and tables

# Geometric altitude (km)
altitude_km = np.array([
    120, 130, 140, 150, 160, 170, 180,
    190, 200, 210, 220, 230, 240, 250,
    260, 270, 280, 290, 300, 310, 320,
    330, 340, 350, 360, 370, 380, 390,
    400, 450, 500, 550, 600, 650, 700,
    750, 800, 850, 900, 950, 1000
], dtype=float)

# Atmospheric density (kg/m^3)
density_values = np.array([
    2.2199e-8, 8.1494e-9, 3.8313e-9,
    2.0752e-9, 1.2336e-9, 7.8155e-10,
    5.1940e-10, 3.5807e-10, 2.5407e-10,
    1.8456e-10, 1.3671e-10, 1.0290e-10,
    7.8550e-11, 6.0706e-11, 4.7420e-11,
    3.7382e-11, 2.9710e-11, 2.3783e-11,
    1.9159e-11, 1.5519e-11, 1.2634e-11,
    1.0333e-11, 8.4886e-12, 7.0011e-12,
    5.7954e-12, 4.8132e-12, 4.0093e-12,
    3.3484e-12, 2.8028e-12,
    1.1843e-12, 5.2148e-13, 2.3832e-13,
    1.1367e-13, 5.7114e-14, 3.0698e-14,
    1.7900e-14, 1.1358e-14, 7.8223e-15,
    5.7587e-15, 4.4525e-15, 3.5611e-15
])

def density(h):
    """
    Input: altitude h in meters
    Output: density in kg/m^3
    """

    h_km = np.asarray(h) / 1000.0


    if np.any((h_km < 120) or (h_km > 1000)):
        raise ValueError("Altitude outside 120-1000 km", h_km)

    result = np.exp(
        np.interp(h_km, altitude_km, np.log(density_values))
    )
    return float(result) if result.ndim == 0 else result
def drag_area(theta):
    #theta comes in radians
    theta=theta%(2*np.pi)# taking the modulus
    
    
    if theta<0:
        raise ValueError(" theta must be beetween 0 and 2 pi")
    elif theta<np.pi:
        return satelite_core_area+ panel_area*(np.sin(theta)**2)
    else:
        return satelite_core_area
    


In [ ]:
def derivative_func(values):
    x,y= values[0], values[1]
    vx,vy=values[2], values[3]
    
    v_mag=np.sqrt(vx*vx+vy*vy)
    r=np.sqrt(x*x+y*y)
    h=r-r_earth
    theta=np.atan2(y,x)+2*np.pi

    
    
    rho= density(np.clip(h, 120e3, 1000e3))
    area= drag_area(theta)
    
    ax= -mu*x/(r**3)-CD*area*rho*v_mag*vx/(2*satelite_mass)
    ay= -mu*y/(r**3)-CD*area*rho*v_mag*vy/(2*satelite_mass)
    
    return np.array([vx,vy,ax,ay])

In [ ]:
# Method implementation

def rk4(current_vals,t_end,dt):
    #does rk on current_values from 0 to t_end
    n=int(np.ceil(t_end/dt))
    u_cur=current_vals
    
    
    for i in range(n):
        k1= derivative_func(u_cur)*dt
        k2=dt*derivative_func( u_cur+k1/2)
        k3= dt* derivative_func(u_cur+k2/2)
        k4= dt*derivative_func(u_cur+k3)
        totaldif= (k1+2*k2+2*k3+k4)/6
        u_cur=u_cur+ totaldif
        
    return u_cur

def check_if_below_limit( u_cur):
    x,y=u_cur[0],u_cur[1]
    r=np.sqrt(x*x+y*y)-r_earth
    
    return r< cut_off_limit
    
    
def find_until_crash( u_cur,dt):
    time=0
    max_time=10*365*24*60*60
    while not check_if_below_limit(u_cur):
        u_cur=rk4(u_cur,dt,dt)    
        time=time+dt    
        if time> max_time:
           raise ValueError(" orbit took too long to decay")
    return u_cur,time

In [ ]:
starting_altitude = np.array([400,450, 500]) * 1e3  # meters

r0 = r_earth + starting_altitude

u_i = np.column_stack((
    r0,
    np.zeros_like(r0),
    np.zeros_like(r0),
    np.sqrt(mu / r0)
))

times_to_test=[15,30,60]
time=np.zeros((len(u_i),len(times_to_test)))


for i in range(len(u_i)):
    u=u_i[i]
    for j in range(len(times_to_test)):
        dt= times_to_test[j]

        x,time[i][j]= find_until_crash(u, dt)
        time[i][j]=time[i][j]/(3600*24*30)
print("months",time)

In [ ]:
print("months",time)
"""data from orbits of 400,450,500km with dt=15,30,60 each column is for one 1dt value


months [[ 4.82445023  4.81798611  4.62701389]
 [12.21586227 12.18149306 11.21092593]
 [29.25572917 29.08474537 24.71030093]





"""

In [ ]:

# ============================================================
# AE 370 - RK4 VERIFICATION AND CONVERGENCE TESTS
# Uses existing notebook functions:
#   derivative_func(), rk4()
# Uses existing constants:
#   mu, r_earth, CD, cut_off_limit
#
# No external ODE integrators.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

# ----------------------------
# USER TEST SETTINGS
# ----------------------------

TEST_ALTITUDE_KM = 400
DT_TEST = np.array([240, 120, 60, 30, 15], dtype=float)

# Actual orbital-decay convergence study
DECAY_ALTITUDES_KM = [200, 250,300]
DECAY_DT = [60, 30, 15, 7.5]

# Higher altitudes take longer to simulate.

MONTH = 30 * 24 * 3600


# ----------------------------
# COMMON HELPER FUNCTIONS
# ----------------------------

def orbital_radius(u):
    return np.hypot(u[0], u[1])


def altitude(u):
    return orbital_radius(u) - r_earth


def orbital_energy(u):
    x, y, vx, vy = u
    return 0.5 * (vx**2 + vy**2) - mu / np.hypot(x, y)


def initial_circular_state(h):
    r0 = r_earth + h

    return np.array([
        r0,
        0.0,
        0.0,
        np.sqrt(mu / r0)
    ])


def integrate_to_time(u0, t_end, dt):
    """
    Shortens the last step to reach t_end exactly.
    """
    u = np.array(u0, dtype=float).copy()

    n = int(np.floor(t_end / dt))

    for i in range(n):
        u = rk4(u, dt, dt)

    remainder = t_end - n * dt

    if remainder > 1e-9:
        u = rk4(u, remainder, remainder)

    return u


# ============================================================
# TEST 1 - EXACT CIRCULAR ORBIT (CD = 0)
# ============================================================

print("\n========== TEST 1: EXACT SOLUTION ==========")

original_CD = CD
CD = 0.0

try:
    r0 = r_earth + TEST_ALTITUDE_KM * 1000
    omega = np.sqrt(mu / r0**3)
    T = 2 * np.pi / omega

    u0 = initial_circular_state(TEST_ALTITUDE_KM * 1000)

    def exact_orbit(t):
        return np.array([
            r0 * np.cos(omega*t),
            r0 * np.sin(omega*t),
            -r0 * omega * np.sin(omega*t),
            r0 * omega * np.cos(omega*t)
        ])

    exact = exact_orbit(T)

    position_errors = []
    velocity_errors = []

    print(f"Orbital period: {T:.4f} seconds")
    print()
    print("dt (s)   Position error (m)   Velocity error (m/s)")

    for dt in DT_TEST:
        numerical = integrate_to_time(u0, T, dt)

        pos_error = np.linalg.norm(
            numerical[:2] - exact[:2]
        )

        vel_error = np.linalg.norm(
            numerical[2:] - exact[2:]
        )

        position_errors.append(pos_error)
        velocity_errors.append(vel_error)

        print(f"{dt:6.1f}   {pos_error:18.8e}   {vel_error:18.8e}")

    position_errors = np.array(position_errors)
    velocity_errors = np.array(velocity_errors)

    # ========================================================
    # TEST 2 - OBSERVED ORDER OF CONVERGENCE
    # ========================================================

    print("\n========== TEST 2: RK4 ORDER ==========")
    print("Expected global order: p = 4")
    print()
    print("dt -> dt/2       Observed order")

    observed_orders = []

    for i in range(len(DT_TEST)-1):
        p = np.log(
            position_errors[i] / position_errors[i+1]
        ) / np.log(DT_TEST[i] / DT_TEST[i+1])

        observed_orders.append(p)

        print(
            f"{DT_TEST[i]:5.1f} -> {DT_TEST[i+1]:5.1f}"
            f"       {p:.5f}"
        )

    plt.figure(figsize=(7, 5))

    plt.loglog(
        DT_TEST,
        position_errors,
        "o-",
        label="Measured RK4 position error"
    )

    # Fourth-order reference line
    reference = position_errors[0] * (
        DT_TEST / DT_TEST[0]
    )**4

    plt.loglog(
        DT_TEST,
        reference,
        "--",
        label="Fourth-order reference"
    )

    plt.xlabel("Timestep (s)")
    plt.ylabel("Position error after one orbit (m)")
    plt.title("RK4 Convergence - Exact Circular Orbit")
    plt.legend()
    plt.grid(True, which="both")
    plt.show()

    # ========================================================
    # TEST 3 - ENERGY CONSERVATION WITH NO DRAG
    # ========================================================

    print("\n========== TEST 3: ENERGY CONSERVATION ==========")

    energy_dt = [120, 60, 30, 15]
    n_orbits = 10

    E0 = orbital_energy(u0)

    print("dt (s)   Maximum relative energy deviation")

    energy_errors = []

    for dt in energy_dt:
        u = u0.copy()
        max_error = 0.0

        n_steps = int(np.ceil(n_orbits * T / dt))

        for i in range(n_steps):
            u = rk4(u, dt, dt)

            E = orbital_energy(u)

            error = abs((E - E0) / E0)
            max_error = max(max_error, error)

        energy_errors.append(max_error)

        print(f"{dt:6.1f}   {max_error:.8e}")

    plt.figure(figsize=(7, 5))
    plt.loglog(energy_dt, energy_errors, "o-")
    plt.xlabel("Timestep (s)")
    plt.ylabel("Maximum relative energy deviation")
    plt.title("Drag-Free Orbital Energy Conservation")
    plt.grid(True, which="both")
    plt.show()

finally:
    # Restore physical drag coefficient, even if a test fails.
    CD = original_CD


# ============================================================
# TEST 4 - ORBITAL DECAY TIMESTEP CONVERGENCE
# ============================================================

print("\n========== TEST 4: ORBITAL DECAY ==========")
print(f"Drag coefficient restored: CD = {CD}")


def find_crossing(u_old, u_new, h_limit):
    """
    Detect a downward altitude crossing.
    Linearly interpolate time fraction and state.

    Returns (fraction, state) or None.
    """

    h_old = altitude(u_old)
    h_new = altitude(u_new)

    if h_old > h_limit >= h_new:
        fraction = (h_old - h_limit) / (h_old - h_new)

        u_cross = u_old + fraction * (u_new - u_old)

        return fraction, u_cross

    return None


def simulate_decay(u0, dt):
    """
    Integrates using your RK4 code.

    Outputs:
        time to 180 km
        time to 120 km
        speed at 120 km

    Times are in seconds; speed is in m/s.
    """

    u = np.array(u0, dtype=float).copy()
    t = 0.0

    t180 = None
    t120 = None
    v120 = None

    max_time = 10 * 365 * 24 * 3600

    while t < max_time:

        u_new = rk4(u, dt, dt)

        if t180 is None:
            crossing180 = find_crossing(u, u_new, 180e3)

            if crossing180 is not None:
                fraction, u180 = crossing180
                t180 = t + fraction * dt

        crossing120 = find_crossing(
            u, u_new, cut_off_limit
        )

        if crossing120 is not None:
            fraction, u120 = crossing120

            t120 = t + fraction * dt
            v120 = np.linalg.norm(u120[2:])

            break

        u = u_new
        t += dt

    if t120 is None:
        raise RuntimeError("120 km not reached within 10 years")

    return t180, t120, v120


results = {}

for h0_km in DECAY_ALTITUDES_KM:

    print(f"\nInitial altitude: {h0_km} km")

    u0_decay = initial_circular_state(h0_km * 1000)

    results[h0_km] = {}

    print("dt (s)  T180 (months)  T120 (months)  V120 (m/s)")

    for dt in DECAY_DT:

        t180, t120, v120 = simulate_decay(u0_decay, dt)

        results[h0_km][dt] = (t180, t120, v120)

        print(
            f"{dt:6.1f}  "
            f"{t180/MONTH:13.8f}  "
            f"{t120/MONTH:13.8f}  "
            f"{v120:12.5f}"
        )


# ============================================================
# TEST 5 - RELATIVE ERRORS FOR PROJECT OUTPUTS
# ============================================================

print("\n========== TEST 5: RELATIVE ERRORS ==========")

# Smallest timestep is the provisional reference.
# It is NOT an exact answer.
reference_dt = min(DECAY_DT)

for h0_km in DECAY_ALTITUDES_KM:

    t180_ref, t120_ref, v120_ref = results[h0_km][reference_dt]

    print(f"\nInitial altitude: {h0_km} km")
    print("dt (s)   Error T180 (%)   Error T120 (%)   Error V120 (%)")

    for dt in DECAY_DT:

        t180, t120, v120 = results[h0_km][dt]

        e180 = 100 * abs(t180-t180_ref) / abs(t180_ref)
        e120 = 100 * abs(t120-t120_ref) / abs(t120_ref)
        ev = 100 * abs(v120-v120_ref) / abs(v120_ref)

        print(f"{dt:6.1f}   {e180:14.6f}   {e120:14.6f}   {ev:14.6f}")


# ============================================================
# TEST 6 - PLOT ERRORS VS TIMESTEP
# ============================================================

for h0_km in DECAY_ALTITUDES_KM:

    ref_t180, ref_t120, ref_v120 = results[h0_km][reference_dt]

    dt_plot = []
    e180_plot = []
    e120_plot = []
    ev_plot = []

    for dt in DECAY_DT:
        if dt == reference_dt:
            continue

        t180, t120, v120 = results[h0_km][dt]

        dt_plot.append(dt)
        e180_plot.append(abs(t180-ref_t180)/abs(ref_t180))
        e120_plot.append(abs(t120-ref_t120)/abs(ref_t120))
        ev_plot.append(abs(v120-ref_v120)/abs(ref_v120))

    plt.figure(figsize=(7, 5))

    plt.loglog(dt_plot, e180_plot, "o-", label="Time to 180 km")
    plt.loglog(dt_plot, e120_plot, "s-", label="Time to 120 km")
    plt.loglog(dt_plot, ev_plot, "^-", label="Speed at 120 km")

    plt.axhline(0.001, linestyle="--", label="0.1% criterion")

    plt.xlabel("Timestep (s)")
    plt.ylabel("Relative error")
    plt.title(f"Orbital Decay Convergence - {h0_km} km")
    plt.legend()
    plt.grid(True, which="both")
    plt.show()

print("\nAll verification tests completed.")



# ============================================================
# TEST 7 - DRAG ENERGY BALANCE VERIFICATION
# Uses your existing rk4(), density(), drag_area()
# and project constants.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

h0 = 200e3
r0 = r_earth + h0

T = 2 * np.pi * np.sqrt(r0**3 / mu)

u0 = np.array([r0, 0.0, 0.0, np.sqrt(mu/r0)])

dt_values = [120, 60, 30, 15, 7.5]

def specific_energy(u):
    x, y, vx, vy = u
    r = np.hypot(x, y)
    return 0.5*(vx**2 + vy**2) - mu/r

def drag_power(u):
    x, y, vx, vy = u

    r = np.hypot(x, y)
    h = r - r_earth
    v = np.hypot(vx, vy)
    theta = np.arctan2(y, x) % (2*np.pi)

    # Same atmospheric boundary handling as your model
    rho = density(np.clip(h, 120e3, 1000e3))
    A = drag_area(theta)

    # Specific power: W/kg = m^2/s^3
    return -CD * A * rho * v**3 / (2*satelite_mass)


def rk4_energy_step(u, dt):
    """
    Advance both orbital state and cumulative drag energy
    using classical RK4.

    The state uses your existing derivative_func().
    """
    k1 = derivative_func(u)
    p1 = drag_power(u)

    u2 = u + dt*k1/2
    k2 = derivative_func(u2)
    p2 = drag_power(u2)

    u3 = u + dt*k2/2
    k3 = derivative_func(u3)
    p3 = drag_power(u3)

    u4 = u + dt*k3
    k4 = derivative_func(u4)
    p4 = drag_power(u4)

    u_new = u + dt*(k1 + 2*k2 + 2*k3 + k4)/6

    energy_drag = dt*(p1 + 2*p2 + 2*p3 + p4)/6

    return u_new, energy_drag


errors = []
state_losses = []
drag_losses = []

print("========== TEST 7: DRAG ENERGY BALANCE ==========")
print("One orbit at 200 km")
print()
print("dt(s)   State energy loss (J/kg)   Drag integral (J/kg)   Rel. mismatch")

for dt in dt_values:

    u = u0.copy()
    E0 = specific_energy(u)

    drag_energy_total = 0.0
    elapsed = 0.0

    # Integrate to exactly one initial orbital period
    while elapsed < T:

        step = min(dt, T - elapsed)

        u, energy_step = rk4_energy_step(u, step)

        drag_energy_total += energy_step
        elapsed += step

    state_energy_change = specific_energy(u) - E0

    # Relative energy-balance mismatch
    error = abs(
        state_energy_change - drag_energy_total
    ) / abs(drag_energy_total)

    errors.append(error)
    state_losses.append(state_energy_change)
    drag_losses.append(drag_energy_total)

    print(
        f"{dt:5.1f}   "
        f"{state_energy_change:23.8f}   "
        f"{drag_energy_total:20.8f}   "
        f"{error:.8e}"
    )


# ============================================================
# OBSERVED CONVERGENCE ORDER
# ============================================================

print("\nObserved energy-balance convergence order:")

for i in range(len(dt_values)-1):
    p = np.log(errors[i]/errors[i+1]) / np.log(
        dt_values[i]/dt_values[i+1]
    )

    print(
        f"{dt_values[i]} -> {dt_values[i+1]} s: "
        f"p = {p:.4f}"
    )


# ============================================================
# PLOT 1 - ENERGY BALANCE CONVERGENCE
# ============================================================

plt.figure(figsize=(7, 5))

plt.loglog(
    dt_values, errors, "o-",
    label="Energy-balance mismatch"
)

reference = errors[0] * (
    np.array(dt_values)/dt_values[0]
)**4

plt.loglog(
    dt_values, reference, "--",
    label="Fourth-order reference"
)

plt.xlabel("Timestep (s)")
plt.ylabel("Relative energy-balance mismatch")
plt.title("Atmospheric Drag Energy Balance")
plt.grid(True, which="both")
plt.legend()
plt.show()


# ============================================================
# PLOT 2 - COMPARE TWO ENERGY CALCULATIONS
# ============================================================

plt.figure(figsize=(7, 5))

plt.plot(
    dt_values, state_losses, "o-",
    label="Energy change from orbital state"
)

plt.plot(
    dt_values, drag_losses, "s--",
    label="Integrated atmospheric drag power"
)

plt.xlabel("Timestep (s)")
plt.ylabel("Specific energy change (J/kg)")
plt.title("Independent Calculations of Orbital Energy Loss")
plt.grid(True)
plt.legend()
plt.show()

print("\nDrag energy verification completed.")

original_drag_area = drag_area

def constant_drag_area(theta):
    return satelite_core_area + panel_area / 4

dt_values = [15, 30, 60]
h0 = 250e3
u0 = initial_circular_state(h0)

try:
    drag_area = constant_drag_area

    print("CONSTANT AREA:")
    for dt in dt_values:
        _, t = find_until_crash(u0.copy(), dt)
        print(dt, t / (3600 * 24 * 30))

finally:
    drag_area = original_drag_area

print("\nORIGINAL PIECEWISE AREA:")
for dt in dt_values:
    _, t = find_until_crash(u0.copy(), dt)
    print(dt, t / (3600 * 24 * 30))
    
original_CD = CD
CD = 0

u0 = initial_circular_state(250e3)
E0 = orbital_energy(u0)

duration = 0.144 * 30 * 24 * 3600

try:
    for dt in [15, 30, 60]:
        u = integrate_to_time(u0, duration, dt)
        E = orbital_energy(u)

        print(
            f"dt={dt} s, "
            f"energy drift={E-E0:.6f} J/kg, "
            f"relative drift={abs((E-E0)/E0)*100:.6f}%"
        )
finally:
    CD = original_CD

In [ ]:
original_CD = CD
CD = 0

u0 = initial_circular_state(250e3)
E0 = orbital_energy(u0)

duration = 0.144 * 30 * 24 * 3600

try:
    for dt in [15, 30, 60]:
        u = integrate_to_time(u0, duration, dt)
        E = orbital_energy(u)

        print(
            f"dt={dt} s, "
            f"energy drift={E-E0:.6f} J/kg, "
            f"relative drift={abs((E-E0)/E0)*100:.6f}%"
        )
finally:
    CD = original_CD

In [ ]:
#Based on above code we see that t=15 is a good choice to be accurate

starting_altitude = np.array([150,200,250,300,350]) * 1e3  # meters

r0 = r_earth + starting_altitude

u_i = np.column_stack((
    r0,
    np.zeros_like(r0),
    np.zeros_like(r0),
    np.sqrt(mu / r0)
))

dt=15
time=np.zeros(len(u_i))
for i in range(len(starting_altitude)):
    u=u_i[i]
    x,time[i]= find_until_crash(u,dt)
    
print(time/(3600*24*30))


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

# Use results from your previous simulation
altitude_km = starting_altitude / 1000
lifetime_days = time / (3600 * 24)
lifetime_months = time / (3600 * 24 * 30)

# ============================================================
# PLOT 1 - ORBITAL LIFETIME VS INITIAL ALTITUDE
# ============================================================

plt.figure(figsize=(9, 5))

plt.plot(
    altitude_km,
    lifetime_months,
    "o-",
    linewidth=2,
    markersize=7
)

plt.xlabel("Initial Orbital Altitude (km)")
plt.ylabel("Time to Reach 120 km (months)")
plt.title("Satellite Orbital Lifetime vs Initial Altitude")
plt.grid(True, alpha=0.3)
plt.xticks(altitude_km)

plt.tight_layout()
plt.show()


# ============================================================
# PLOT 2 - LOG SCALE
# ============================================================

plt.figure(figsize=(9, 5))

plt.semilogy(
    altitude_km,
    lifetime_months,
    "o-",
    linewidth=2,
    markersize=7
)

plt.xlabel("Initial Orbital Altitude (km)")
plt.ylabel("Time to Reach 120 km (months, log scale)")
plt.title("Orbital Lifetime vs Initial Altitude - Logarithmic Scale")
plt.xticks(altitude_km)
plt.grid(True, which="both", alpha=0.3)

plt.tight_layout()
plt.show()


# ============================================================
# PLOT 3 - LABELED DATA POINTS
# ============================================================

plt.figure(figsize=(10, 5))

plt.plot(
    altitude_km,
    lifetime_months,
    "o-",
    linewidth=2,
    markersize=8
)

for h, months in zip(altitude_km, lifetime_months):
    plt.annotate(
        f"{months:.3f} mo",
        (h, months),
        textcoords="offset points",
        xytext=(0, 10),
        ha="center"
    )

plt.xlabel("Initial Orbital Altitude (km)")
plt.ylabel("Time to Reach 120 km (months)")
plt.title("Satellite Orbital Decay - RK4 (dt = 15 s)")
plt.xticks(altitude_km)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# ============================================================
# PRINT DATA TABLE
# ============================================================

print("Initial Altitude (km) | Lifetime (days) | Lifetime (months)")
print("-" * 62)

for h, days, months in zip(
    altitude_km, lifetime_days, lifetime_months
):
    print(f"{h:21.0f} | {days:15.4f} | {months:17.6f}")


In [ ]:
# Results helper cell

import numpy as np
import matplotlib.pyplot as plt

def run_orbit(h0_km, dt=15):
    # Initial circular orbit
    r0 = r_earth + h0_km * 1000

    u = np.array([r0, 0, 0, np.sqrt(mu/r0)], dtype=float)

    h_before = h0_km * 1000

    # Maximum steps prevents an endless simulation
    for k in range(600000):

        # Use the original RK4 function
        u_next = rk4(u, dt, dt)

        # Calculate new altitude
        h_after = np.hypot(u_next[0], u_next[1]) - r_earth

        # Stop when we reach 120 km
        if h_after <= cut_off_limit:

            # Interpolate to exactly 120 km
            f = (h_before - cut_off_limit)/(h_before - h_after)
            u_120 = u + f * (u_next - u)

            lifetime_days = (k + f)*dt / 86400
            final_speed = np.hypot(u_120[2], u_120[3])

            return lifetime_days, final_speed

        u = u_next
        h_before = h_after

    raise RuntimeError("Maximum number of steps reached")

In [ ]:

# Section 4.1 - Initial Altitude vs Orbital Lifetime

altitudes_km = np.array([150, 200, 250, 300, 350])

lifetime_days = []
final_speeds = []

dt_results = 15

for h in altitudes_km:
    print(f"Running {h} km...", flush=True)

    days, speed = run_orbit(h, dt_results)

    lifetime_days.append(days)
    final_speeds.append(speed)

# Print results
print("\nAltitude (km) | Lifetime (days) | Lifetime (months)")
for h, days in zip(altitudes_km, lifetime_days):
    print(f"{h:13.0f} | {days:15.4f} | {days/30:17.5f}")

# Plot lifetime vs initial altitude
plt.figure(figsize=(8, 5))

plt.plot(altitudes_km, lifetime_days, "o-")

plt.xlabel("Initial Orbital Altitude (km)")
plt.ylabel("Time to Reach 120 km (days)")
plt.title("Orbital Lifetime vs Initial Altitude")
plt.xticks(altitudes_km)
plt.grid(True)

plt.tight_layout()
plt.savefig("Figure_4_1.png", dpi=200)
plt.show()

In [ ]:
# Section 4.2 - Effect of Solar Panel Configuration

# Save the original area function
saved_area = drag_area

# Four configurations
configurations = [
    ("Retracted (0.5 m²)",
     lambda theta: satelite_core_area),

    ("Variable area",
     saved_area),

    ("Constant mean (1.0 m²)",
     lambda theta: satelite_core_area + panel_area/4),

    ("Maximum area (2.5 m²)",
     lambda theta: satelite_core_area + panel_area)
]

config_names = []
config_lifetimes = []

try:
    for name, area_function in configurations:

        # Temporarily change the projected area model
        drag_area = area_function

        days, speed = run_orbit(250, dt_results)

        config_names.append(name)
        config_lifetimes.append(days)

        print(f"{name}: {days:.4f} days")

finally:
    # Restore the original model
    drag_area = saved_area

# Plot the results
plt.figure(figsize=(9, 5))

plt.bar(config_names, config_lifetimes)

plt.xlabel("Projected Area Configuration")
plt.ylabel("Time to Reach 120 km (days)")
plt.title("Effect of Solar Panel Area on Orbital Lifetime")

plt.xticks(rotation=15, ha="right")
plt.grid(axis="y")

plt.tight_layout()
plt.show()

In [ ]:
# Section 4.3 - Velocity at 120 km

print("Altitude (km) | Speed at 120 km (m/s)")

for h, speed in zip(altitudes_km, final_speeds):
    print(f"{h:13.0f} | {speed:20.2f}")

# Plot final speed against initial altitude
plt.figure(figsize=(8, 5))

plt.plot(altitudes_km, final_speeds, "o-")

plt.xlabel("Initial Orbital Altitude (km)")
plt.ylabel("Speed at 120 km (m/s)")
plt.title("Final Speed vs Initial Altitude")

plt.xticks(altitudes_km)
plt.grid(True)

plt.tight_layout()
plt.show()